# R04 — Rotation and SimCLR Downstream Skin-Lesion Classification
**Kaggle name (first seed):** `lesion-r04-ssl-classification-s42`  
**Required Kaggle notebook outputs:** `lesion-n01-dataset-audit-grouped-splits`, `lesion-r00-protocol-lock`, `lesion-r02-rotation-ssl-s42`, `lesion-r03-simclr-ssl-s42`. Attach the corresponding R02/R03 seed for runs 43–46. The uploaded `.ipynb` files alone do not contain the `.pt` model artifacts.

This notebook **independently fine-tunes both SSL encoders**, one after the other, using all 16,490 labeled training images. The 2,186-image validation split selects checkpoints; the 2,267-image meta-training split is **inference only**. The held-out test split is never loaded. Same head and fine-tuning recipe as R01 for controlled comparison; **no reduced-label budgets**.

Run with a Kaggle GPU; no Internet needed because both encoders are supplied by completed R02/R03 notebook outputs. One seed per execution; change `SEED` only. Every model has independent checkpoints and is resumable from a previously saved R04 notebook output.

In [1]:
import os,sys,time,json,random,hashlib,shutil,gc,platform
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,classification_report
import matplotlib.pyplot as plt

SEED=42
SEEDS=[42,43,44,45,46]
SOURCES=['rotation','simclr']
HEAD_EPOCHS=8
FINETUNE_EPOCHS=35
HEAD_LR=1e-3
FINETUNE_LR=3e-5
BATCH_SIZE=32
EVAL_BATCH=64
WEIGHT_DECAY=1e-4
DROPOUT=.30
PATIENCE=8
AMP=True
WORKERS=min(4,os.cpu_count() or 2)
TIME_LIMIT_HOURS=8.5
MAX_EPOCHS_THIS_RUN=int(os.getenv('R04_MAX_EPOCHS_THIS_RUN','0'))
INPUT_DIR=Path(os.getenv('R04_INPUT_DIR','/kaggle/input'))
OUT=Path(os.getenv('R04_OUTPUT_DIR',f'/kaggle/working/lesion-r04-ssl-classification-s{SEED}' if Path('/kaggle/working').exists() else f'./lesion-r04-ssl-classification-s{SEED}')).resolve()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda' and os.getenv('R04_ALLOW_CPU','0')!='1':raise RuntimeError('Enable a Kaggle GPU for R04')
if SEED not in SEEDS:raise ValueError('SEED must be 42–46')
if MAX_EPOCHS_THIS_RUN<0:raise ValueError('MAX_EPOCHS_THIS_RUN must be >=0')
for folder in ['models','training_state','predictions','reports','tables','figures']:(OUT/folder).mkdir(parents=True,exist_ok=True)
START=time.monotonic()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'

def require(ok,msg):
    if not bool(ok):raise RuntimeError(msg)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(2**20),b''):h.update(block)
    return h.hexdigest()

def save_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,indent=2,allow_nan=False,ensure_ascii=False)+'\n',encoding='utf-8')
    tmp.replace(path)

def save_pt(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');torch.save(data,tmp);tmp.replace(path)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True,warn_only=True)

seed_all(SEED)
print('R04 seed:',SEED,'device:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU test')
print('Output:',OUT,'PyTorch:',torch.__version__)

R04 seed: 42 device: Tesla T4
Output: /kaggle/working/lesion-r04-ssl-classification-s42 PyTorch: 2.11.0+cu130


## 1. Verify frozen N01/R00 inputs and the corresponding R02/R03 encoders

In [2]:
def discover(marker,field,value,override=''):
    roots=[Path(override).expanduser().resolve()] if override else sorted({p.parent.parent for p in INPUT_DIR.rglob(marker)})
    good=[]
    for root in roots:
        f=root/marker
        if not f.is_file():continue
        try:meta=json.loads(f.read_text())
        except (ValueError,OSError):continue
        if meta.get(field)==value:good.append(root)
    good=list(dict.fromkeys(good))
    require(len(good)==1,f'Attach exactly one {value} output containing {marker}; found {good}')
    return good[0]

R00=discover('configs/experiment_protocol.json','name','lesion-r00-protocol-lock',os.getenv('R04_R00_ROOT',''))
N01=discover('reports/protocol.json','project','lesion-n01-dataset-audit-grouped-splits',os.getenv('R04_N01_ROOT',''))
protocol=json.loads((R00/'configs/experiment_protocol.json').read_text())
n01_info=json.loads((N01/'reports/protocol.json').read_text())
checks=json.loads((R00/'reports/artifact_checksums.json').read_text())
for rel in ['configs/experiment_protocol.json','splits/train.csv','splits/validation.csv','splits/meta_train.csv']:
    require(sha256(R00/rel)==checks.get(rel),f'R00 checksum mismatch: {rel}')
require(sha256(N01/'splits/experiment_master_manifest.csv')==protocol['n01_manifest_sha256'],'Wrong N01 master manifest')
require(n01_info.get('manifest_sha256')==protocol['n01_manifest_sha256'],'N01/R00 protocol mismatch')
require(protocol['status']=='LOCKED' and protocol['classes']==CLASSES,'Incorrect locked classes/protocol')
require(protocol['random_seeds']==SEEDS and protocol['use_reduced_label_budgets'] is False,'Incorrect seed/budget protocol')
require(protocol['data_access']['ssl_finetuning']==['train'] and protocol['data_access']['ensemble_meta_fit']==['meta_train'],'Unexpected R00 data-access rules')
require(protocol['image_shape']==[224,224,3],'Expected N01 224×224 images')

frames={}
for split in ['train','validation','meta_train']:
    f=pd.read_csv(R00/'splits'/f'{split}.csv',dtype='string',keep_default_na=False)
    require(len(f)==protocol['split_counts'][split] and f.image_id.is_unique,f'Wrong {split} records')
    require(f.split.eq(split).all(),'Mixed split identity')
    require(f[['image_id','image_relpath','group_id','source_sha256','label']].ne('').all().all(),'Missing identity field')
    require(f.label.isin(CLASSES).all(),'Unknown label')
    require(f.class_index.eq(f.label.map({v:str(i) for i,v in enumerate(CLASSES)})).all(),'Class order/index mismatch')
    for rel in f.image_relpath:
        q=Path(rel)
        require(not q.is_absolute() and '..' not in q.parts and q.parts[:1]==('images',),f'Unsafe image path: {rel}')
        require((N01/q).is_file(),f'Missing N01 image {rel}')
    frames[split]=f.sort_values('image_id').reset_index(drop=True)
for a,b in [('train','validation'),('train','meta_train'),('meta_train','validation')]:
    for field in ['image_id','group_id','source_sha256']:
        require(set(frames[a][field]).isdisjoint(frames[b][field]),f'{a}/{b} overlap: {field}')
require([len(frames[s]) for s in ['train','validation','meta_train']]==[16490,2186,2267],'Unexpected N01 split counts')
master=pd.read_csv(N01/'splits/experiment_master_manifest.csv',dtype='string',keep_default_na=False)
expected=master.loc[master.split.eq('meta_train'),['image_id','class_index','group_id','image_relpath']].sort_values('image_id').reset_index(drop=True)
observed=frames['meta_train'][['image_id','class_index','group_id','image_relpath']]
require(expected.equals(observed),'Meta-training data differs from locked N01 master')

pretext={};pretext_metadata={}
for source,stage,stem in [('rotation','R02','lesion-r02-rotation-ssl'),('simclr','R03','lesion-r03-simclr-ssl')]:
    override=os.getenv(f'R04_{stage}_ROOT','').strip()
    roots=[Path(override).expanduser().resolve()] if override else sorted({p.parent.parent for p in INPUT_DIR.rglob('models/encoder_best.pt')})
    good=[]
    for root in roots:
        met=root/'reports/final_metrics.json';cfg=root/'reports/config.json';enc=root/'models/encoder_best.pt'
        if not all(p.is_file() for p in (met,cfg,enc)):continue
        try:report=json.loads(met.read_text());conf=json.loads(cfg.read_text())
        except (OSError,ValueError):continue
        if report.get('stage')==stage and report.get('seed')==SEED and report.get('status')=='COMPLETED' and conf.get('seed')==SEED:good.append(root)
    good=list(dict.fromkeys(good))
    require(len(good)==1,f'Attach exactly one completed {stem}-s{SEED} Kaggle output; found: {good}')
    root=good[0];path=root/'models/encoder_best.pt';obj=torch.load(path,map_location='cpu',weights_only=False)
    cfg=json.loads((root/'reports/config.json').read_text())
    require(obj.get('stage')==stage and obj.get('seed')==SEED,'Encoder stage/seed mismatch')
    require(obj.get('signature')==cfg.get('config_sha256'),'R02/R03 checkpoint signature mismatch')
    require(obj.get('r00_protocol_sha256')==checks['configs/experiment_protocol.json'],'Encoder trained against different R00 protocol')
    require(obj.get('n01_manifest_sha256')==protocol['n01_manifest_sha256'],'Encoder trained on different N01 manifest')
    require(obj.get('feature_dim')==2048 and obj.get('fc')=='Identity','Unexpected pretrained encoder specification')
    require(isinstance(obj.get('encoder_state'),dict) and 'conv1.weight' in obj['encoder_state'],'Missing ResNet50 encoder weights')
    pretext[source]={'stage':stage,'root':root,'path':path,'sha256':sha256(path),'signature':obj['signature'],'best_epoch':obj.get('pretext_best_epoch')}
    pretext_metadata[source]={'stage':stage,'seed':SEED,'best_pretext_epoch':obj.get('pretext_best_epoch'),
                              'checkpoint_sha256':pretext[source]['sha256'],'pretext_config_sha256':obj['signature'],'checkpoint_root':str(root)}
    del obj
save_json(OUT/'reports/input_audit.json',{'seed':SEED,'N01':str(N01),'R00':str(R00),'n01_manifest_sha256':protocol['n01_manifest_sha256'],
 'r00_protocol_sha256':checks['configs/experiment_protocol.json'],'split_counts':{s:len(f) for s,f in frames.items()},
 'pretext':pretext_metadata,'test_loaded':False,'meta_train_training_prohibited':True})
print('Verified N01 train / validation / meta:',*[len(frames[s]) for s in ['train','validation','meta_train']])
print('Encoder checkpoints:',{s:str(x['path']) for s,x in pretext.items()})

Verified N01 train / validation / meta: 16490 2186 2267
Encoder checkpoints: {'rotation': '/kaggle/input/notebooks/sudiptoroy007/lesion-r02-rotation-ssl-s42/lesion-r02-rotation-ssl-s42/models/encoder_best.pt', 'simclr': '/kaggle/input/notebooks/sudiptoroy007/lesion-r03-simclr-ssl-s42/lesion-r03-simclr-ssl-s42/models/encoder_best.pt'}


## 2. Matched downstream recipe (same as R01)

In [3]:
class RightAngle:
    def __call__(self,img):
        k=random.randrange(4)
        return img.rotate(k*90) if k else img

NORM=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
TRAIN_T=transforms.Compose([transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(),RightAngle(),transforms.ToTensor(),NORM])
EVAL_T=transforms.Compose([transforms.ToTensor(),NORM])

class LesionImages(Dataset):
    def __init__(self,frame,training=False):
        self.paths=[N01/Path(v) for v in frame.image_relpath]
        self.y=frame.class_index.astype(int).to_numpy(dtype=np.int64)
        self.transform=TRAIN_T if training else EVAL_T
    def __len__(self):return len(self.y)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:x=self.transform(im.convert('RGB'))
        return x,int(self.y[i])

def worker_seed(worker_id):
    seed=torch.initial_seed()%2**32;random.seed(seed);np.random.seed(seed)

loader_options={'num_workers':WORKERS,'pin_memory':DEVICE.type=='cuda'}
if WORKERS:loader_options['prefetch_factor']=2
train_dataset=LesionImages(frames['train'],True)
val_loader=DataLoader(LesionImages(frames['validation']),batch_size=EVAL_BATCH,shuffle=False,**loader_options)
meta_loader=DataLoader(LesionImages(frames['meta_train']),batch_size=EVAL_BATCH,shuffle=False,**loader_options)

def train_loader_for(phase,epoch):
    generator=torch.Generator().manual_seed(SEED*100000+epoch+(0 if phase=='head' else 10000))
    return DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=generator,
                      worker_init_fn=worker_seed,**loader_options)

counts=np.bincount(frames['train'].class_index.astype(int).to_numpy(),minlength=len(CLASSES))
require(np.all(counts>0),'Training partition must contain all eight classes')
weights=np.sqrt(len(train_dataset)/counts);weights=.99*weights/weights.max()
CLASS_WEIGHTS=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
pd.DataFrame({'class_index':range(8),'class_name':CLASSES,'train_count':counts,'loss_weight':weights}).to_csv(OUT/'tables/train_class_weights.csv',index=False)

class SSLClassifier(nn.Module):
    def __init__(self,encoder_state):
        super().__init__()
        self.encoder=resnet50(weights=None);self.encoder.fc=nn.Identity()
        self.encoder.load_state_dict(encoder_state,strict=True)
        self.dropout=nn.Dropout(DROPOUT)
        self.classifier=nn.Linear(2048,len(CLASSES))
    def forward(self,x):return self.classifier(self.dropout(self.encoder(x)))

def set_phase(model,phase):
    for p in model.encoder.parameters():p.requires_grad=(phase=='finetune')
    for m in model.encoder.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm):
            for p in m.parameters():p.requires_grad=False
    for p in model.classifier.parameters():p.requires_grad=True

def train_mode(model,phase):
    model.train()
    if phase=='head':model.encoder.eval()
    else:
        for m in model.encoder.modules():
            if isinstance(m,nn.modules.batchnorm._BatchNorm):m.eval()

@torch.inference_mode()
def evaluate(model,loader,return_predictions=False):
    model.eval();loss_sum=0.;n=0;labels=[];logits=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):z=model(x)
        loss_sum+=float(F.cross_entropy(z.float(),y,reduction='sum').item());n+=len(y)
        labels.append(y.cpu().numpy());logits.append(z.float().cpu().numpy())
    y=np.concatenate(labels);z=np.concatenate(logits)
    proba=torch.softmax(torch.from_numpy(z),dim=1).numpy()
    pred=proba.argmax(1)
    metrics={'accuracy':float(accuracy_score(y,pred)),'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
             'macro_f1':float(f1_score(y,pred,average='macro',labels=np.arange(8),zero_division=0)),
             'nll':float(loss_sum/n),'n_images':int(n)}
    return (metrics,y,z,proba) if return_predictions else metrics

print('Training:',len(train_dataset),'Validation:',len(val_loader.dataset),'Meta-training inference:',len(meta_loader.dataset))
print('Trainable head params:',2048*8+8,'full-training samples:',len(train_dataset))

Training: 16490 Validation: 2186 Meta-training inference: 2267
Trainable head params: 16392 full-training samples: 16490


## 3. Train Rotation then SimCLR; independent best/last checkpoints and per-model histories

In [4]:
def restore_rng(state):
    random.setstate(state['python_rng']);np.random.set_state(state['numpy_rng']);torch.set_rng_state(state['torch_rng'])
    if DEVICE.type=='cuda' and state['cuda_rng']:torch.cuda.set_rng_state_all(state['cuda_rng'])

def snapshot(model,source,signature,phase,next_epoch,opt=None,sched=None,scaler=None,bad=0,best_key=None,status='PAUSED'):
    return {'stage':'R04','source':source,'seed':SEED,'signature':signature,'status':status,
            'phase':phase,'next_epoch':int(next_epoch),'bad_epochs':int(bad),'best_key':best_key,
            'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'optimizer':opt.state_dict() if opt else None,'scheduler':sched.state_dict() if sched else None,
            'scaler':scaler.state_dict() if scaler else None,
            'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
            'cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else []}

def find_resume(source):
    local=OUT/'training_state'/source/'last.pt'
    if local.is_file():return local
    explicit=os.getenv(f'R04_RESUME_{source.upper()}','').strip()
    if explicit:
        path=Path(explicit).expanduser().resolve()
        require(path.is_file(),f'R04 resume file not found: {path}')
        return path
    paths=[p for p in INPUT_DIR.rglob(f'training_state/{source}/last.pt')
           if f'lesion-r04-ssl-classification-s{SEED}' in str(p)] if INPUT_DIR.exists() else []
    require(len(paths)<=1,f'Multiple R04 {source} resume inputs: {paths}')
    return paths[0] if paths else None

def train_epoch(model,source,phase,epoch,optimizer,scaler):
    train_mode(model,phase);loss_sum=0.;correct=0;n=0
    for batch_i,(x,y) in enumerate(train_loader_for(phase,epoch)):
        if batch_i%32==0 and time.monotonic()-START>TIME_LIMIT_HOURS*3600:
            raise TimeoutError('Kaggle runtime guard reached within an epoch; last completed checkpoint remains valid')
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
            logits=model(x);loss=(F.cross_entropy(logits.float(),y,reduction='none')*CLASS_WEIGHTS[y]).mean()
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
        scaler.step(optimizer);scaler.update()
        n+=len(y);loss_sum+=float(loss.detach().item())*len(y);correct+=int((logits.argmax(1)==y).sum().item())
    require(n==len(train_dataset),'Incomplete training epoch')
    return float(loss_sum/n),float(correct/n)

def model_paths(source):
    result={k:OUT/v/source for k,v in [('models','models'),('training_state','training_state'),('tables','tables'),
                                       ('predictions','predictions'),('reports','reports'),('figures','figures')]}
    for p in result.values():p.mkdir(parents=True,exist_ok=True)
    return result

def train_one(source):
    paths=model_paths(source)
    cfg={'stage':'R04','name':'lesion-r04-ssl-classification','source':source,'seed':SEED,
         'pretext_stage':pretext[source]['stage'],'pretext_checkpoint_sha256':pretext[source]['sha256'],
         'pretext_config_sha256':pretext[source]['signature'],
         'n01_manifest_sha256':protocol['n01_manifest_sha256'],
         'r00_protocol_sha256':checks['configs/experiment_protocol.json'],
         'architecture':'torchvision.models.resnet50(fc=Identity)+Dropout0.3+Linear(2048,8)',
         'classes':CLASSES,'train_images':len(frames['train']),'validation_images':len(frames['validation']),
         'meta_training_inference_only':True,'batch_size':BATCH_SIZE,'eval_batch':EVAL_BATCH,
         'head_epochs':HEAD_EPOCHS,'finetune_epochs':FINETUNE_EPOCHS,'head_lr':HEAD_LR,
         'finetune_lr':FINETUNE_LR,'patience':PATIENCE,'weight_decay':WEIGHT_DECAY,
         'loss':'weighted_CE, sqrt inverse class prevalence, max weight=0.99',
         'augmentation':'R01 matched H/V flips + right-angle rotation + ImageNet RGB normalization',
         'optimizer':'AdamW','scheduler':'ReduceLROnPlateau(finetune val NLL, factor=.5, patience=3, min_lr=1e-7)',
         'bn_policy':'Freeze BN affine and running statistics during downstream training',
         'checkpoint_selection':'validation balanced_accuracy, then macro_f1, then accuracy',
         'test_access':False}
    signature=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()
    save_json(paths['reports']/'config.json',{**cfg,'config_sha256':signature})
    best=paths['models']/'best.pt';last=paths['training_state']/'last.pt';hist=paths['tables']/'training_history.csv'
    seed_all(SEED)
    resume_path=find_resume(source);state=None
    if resume_path:
        state=torch.load(resume_path,map_location='cpu',weights_only=False)
        require(state.get('stage')=='R04' and state.get('source')==source and state.get('seed')==SEED and state.get('signature')==signature,
                f'R04 {source} resume signature/seed mismatch')
        if resume_path!=last:
            shutil.copy2(resume_path,last)
            former=resume_path.parent.parent.parent
            for rel in [f'models/{source}/best.pt',f'tables/{source}/training_history.csv',
                        f'models/{source}/encoder_best.pt',f'reports/{source}/final_metrics.json',
                        f'predictions/{source}/validation.npz',f'predictions/{source}/meta_train.npz']:
                src=former/rel;dst=OUT/rel
                if src.is_file():dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst)
        if state['phase']!='head' or state['next_epoch']>0:
            require(hist.is_file(),'Resumed R04 training history is missing')
    pre=torch.load(pretext[source]['path'],map_location='cpu',weights_only=False)
    model=SSLClassifier(pre['encoder_state']).to(DEVICE)
    del pre
    if state:
        model.load_state_dict(state['model'])
    h=pd.read_csv(hist).to_dict('records') if hist.exists() else []
    phases=[('head',HEAD_EPOCHS,HEAD_LR),('finetune',FINETUNE_EPOCHS,FINETUNE_LR)]
    current=state['phase'] if state else 'head';next_ep=int(state['next_epoch']) if state else 0
    bad=int(state['bad_epochs']) if state else 0
    best_key=tuple(state.get('best_key') or [-1.,-1.,-1.]) if state else (-1.,-1.,-1.)
    status='COMPLETED' if current=='done' else 'PAUSED'
    epoch_budget=0
    for phase,epochs,lr in phases:
        if current=='done':break
        if [x[0] for x in phases].index(phase)<[x[0] for x in phases].index(current):continue
        set_phase(model,phase)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=WEIGHT_DECAY)
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=3,min_lr=1e-7) if phase=='finetune' else None
        first=next_ep if current==phase else 0
        scaler=torch.amp.GradScaler('cuda',enabled=AMP and DEVICE.type=='cuda')
        if state and phase==current:
            if state['optimizer'] is not None:
                optimizer.load_state_dict(state['optimizer'])
                if scheduler is not None and state['scheduler'] is not None:scheduler.load_state_dict(state['scheduler'])
                if state.get('scaler') is not None:scaler.load_state_dict(state['scaler'])
            restore_rng(state)
        if phase=='finetune' and first==0:bad=0;best_key=(-1.,-1.,-1.)
        print(f'R04 {source} phase={phase} epoch={first+1}/{epochs} trainable={sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
        phase_done=first>=epochs or (phase=='finetune' and bad>=PATIENCE)
        for epoch in range(first if not phase_done else epochs,epochs):
            if time.monotonic()-START>TIME_LIMIT_HOURS*3600 or (MAX_EPOCHS_THIS_RUN and epoch_budget>=MAX_EPOCHS_THIS_RUN):break
            t=time.monotonic()
            try:tr_loss,tr_acc=train_epoch(model,source,phase,epoch,optimizer,scaler)
            except TimeoutError as e:print(str(e));break
            val=evaluate(model,val_loader)
            if scheduler:scheduler.step(val['nll'])
            key=(val['balanced_accuracy'],val['macro_f1'],val['accuracy'])
            improved=phase=='finetune' and key>best_key
            if phase=='finetune':
                if improved:
                    best_key=key;bad=0
                    save_pt(best,{'stage':'R04','source':source,'seed':SEED,'signature':signature,
                                  'phase':phase,'epoch':epoch+1,'validation_metrics':val,'model':model.state_dict(),
                                  'n01_manifest_sha256':protocol['n01_manifest_sha256']})
                else:bad+=1
            row={'seed':SEED,'source':source,'phase':phase,'epoch':epoch+1,'train_weighted_loss':tr_loss,
                 'train_accuracy':tr_acc,'val_nll':val['nll'],'val_balanced_accuracy':val['balanced_accuracy'],
                 'val_macro_f1':val['macro_f1'],'val_accuracy':val['accuracy'],
                 'lr':optimizer.param_groups[0]['lr'],'seconds':time.monotonic()-t,'is_best':int(improved),
                 'train_images_seen':len(train_dataset),'validation_images_seen':len(val_loader.dataset)}
            h.append(row);pd.DataFrame(h).to_csv(hist,index=False)
            save_pt(last,snapshot(model,source,signature,phase,epoch+1,optimizer,scheduler,scaler,bad,list(best_key),'PAUSED'))
            print(f'{source} {phase} {epoch+1:02d}/{epochs} | train CE {tr_loss:.4f} | val BACC {val["balanced_accuracy"]:.4f} | '
                  f'macro-F1 {val["macro_f1"]:.4f} | val NLL {val["nll"]:.4f} | {row["seconds"]:.0f}s'+(' *' if improved else ''))
            epoch_budget+=1
            if phase=='finetune' and bad>=PATIENCE:
                print('Early stopped on validation balanced accuracy');phase_done=True;break
            if epoch==epochs-1:phase_done=True
        if not phase_done:break
        if phase=='head':
            current='finetune';next_ep=0;state=None
            save_pt(last,snapshot(model,source,signature,'finetune',0,None,None,None,0,[-1.,-1.,-1.],'PAUSED'))
            continue
        require(best.is_file(),f'No validation-selected R04 {source} checkpoint')
        current='done';next_ep=0;status='COMPLETED'
        save_pt(last,snapshot(model,source,signature,'done',0,None,None,None,0,list(best_key),'COMPLETED'))
    if current=='done':status='COMPLETED'
    save_json(paths['reports']/'training_status.json',{'stage':'R04','source':source,'seed':SEED,'status':status,
        'history_rows':len(h),'best_checkpoint_present':best.is_file(),'config_sha256':signature,
        'elapsed_seconds':time.monotonic()-START})
    if status=='COMPLETED':
        require(best.is_file(),'Missing best model on completed source')
        best_state=torch.load(best,map_location='cpu',weights_only=False)
        require(best_state['signature']==signature and best_state['source']==source,'Best model signature differs')
        model.load_state_dict(best_state['model'])
        metrics={}
        for split,loader in [('validation',val_loader),('meta_train',meta_loader)]:
            scores,y,z,pr=evaluate(model,loader,True)
            f=frames[split]
            require(np.array_equal(y,f.class_index.astype(int).to_numpy()),f'{source} {split} prediction alignment mismatch')
            require(np.isfinite(pr).all() and np.allclose(pr.sum(axis=1),1,rtol=1e-5,atol=1e-5),'Invalid prediction probabilities')
            np.savez_compressed(paths['predictions']/f'{split}.npz',image_id=f.image_id.to_numpy(dtype=str),
                group_id=f.group_id.to_numpy(dtype=str),y=y.astype(np.int32),logits=z.astype(np.float32),
                proba=pr.astype(np.float32),classes=np.array(CLASSES),seed=np.array(SEED),
                protocol_sha256=np.array(signature),r00_protocol_sha256=np.array(checks['configs/experiment_protocol.json']),
                n01_manifest_sha256=np.array(protocol['n01_manifest_sha256']),
                model_id=np.array(f'{source}_finetuned'),pretext_checkpoint_sha256=np.array(pretext[source]['sha256']))
            metrics[split]=scores
        final_encoder={'stage':'R04','source':source,'seed':SEED,'signature':signature,
                       'encoder_name':'torchvision.models.resnet50','feature_dim':2048,'fc':'Identity',
                       'n01_manifest_sha256':protocol['n01_manifest_sha256'],
                       'r00_protocol_sha256':checks['configs/experiment_protocol.json'],
                       'pretext_encoder_sha256':pretext[source]['sha256'],
                       'classifier_best_epoch':best_state['epoch'],
                       'encoder_state':{k:v.detach().cpu().clone() for k,v in model.encoder.state_dict().items()}}
        save_pt(paths['models']/'encoder_best.pt',final_encoder)
        y=frames['validation'].class_index.astype(int).to_numpy()
        with np.load(paths['predictions']/'validation.npz',allow_pickle=False) as z:prediction=z['proba'].argmax(1)
        cm=confusion_matrix(y,prediction,labels=list(range(8)))
        pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(paths['tables']/'validation_confusion_matrix.csv')
        pd.DataFrame(classification_report(y,prediction,labels=list(range(8)),target_names=CLASSES,
              output_dict=True,zero_division=0)).T.to_csv(paths['tables']/'validation_classification_report.csv')
        history=pd.DataFrame(h)
        fig,axes=plt.subplots(1,2,figsize=(10,3.4))
        for phase,part in history.groupby('phase',sort=False):
            xx=np.arange(len(part))+1+(0 if phase=='head' else HEAD_EPOCHS)
            axes[0].plot(xx,part.val_balanced_accuracy,label=phase)
            axes[1].plot(xx,part.train_weighted_loss,label=f'{phase} train weighted CE')
            axes[1].plot(xx,part.val_nll,label=f'{phase} val NLL')
        axes[0].set(xlabel='Epoch',ylabel='Validation balanced accuracy',title=f'{source.title()} diagnostic classification')
        axes[1].set(xlabel='Epoch',ylabel='Loss',title='Training and validation loss (different weighting)')
        for ax in axes:ax.legend(fontsize=7);ax.grid(alpha=.2)
        fig.tight_layout();fig.savefig(paths['figures']/'training_curves.png',dpi=200,bbox_inches='tight');plt.close(fig)
        fig,ax=plt.subplots(figsize=(6,5));im=ax.imshow(cm,cmap='Blues')
        ax.set_xticks(range(8),CLASSES,rotation=45,ha='right');ax.set_yticks(range(8),CLASSES)
        ax.set(xlabel='Predicted',ylabel='True',title=f'{source.title()} validation confusion')
        fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(paths['figures']/'validation_confusion_matrix.png',dpi=200);plt.close(fig)
        final={'stage':'R04','source':source,'seed':SEED,'status':'COMPLETED','pretext_stage':pretext[source]['stage'],
               'pretext_encoder_sha256':pretext[source]['sha256'],'best_finetune_epoch':best_state['epoch'],
               'selection_metrics':best_state['validation_metrics'],'validation_metrics':metrics['validation'],
               'meta_train_metrics_diagnostic_only':metrics['meta_train'],'train_images':len(frames['train']),
               'validation_images':len(frames['validation']),'meta_training_images':len(frames['meta_train']),
               'n01_manifest_sha256':protocol['n01_manifest_sha256'],'config_sha256':signature,
               'note':'All encoder fitting uses train only; meta_train scores are descriptive, not used to select the classifier.',
               'files':['models/'+source+'/best.pt','models/'+source+'/encoder_best.pt',
                        'training_state/'+source+'/last.pt','tables/'+source+'/training_history.csv',
                        'predictions/'+source+'/validation.npz','predictions/'+source+'/meta_train.npz']}
        save_json(paths['reports']/'final_metrics.json',final)
        print(f'COMPLETED — {source} seed {SEED}: validation BACC {metrics["validation"]["balanced_accuracy"]:.4f}, '
              f'macro-F1 {metrics["validation"]["macro_f1"]:.4f}, best finetune epoch {best_state["epoch"]}')
    else:
        print(f'PAUSED — {source}: attach saved R04 notebook output to resume from training_state/{source}/last.pt')
    del model
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return status

statuses={}
for source in SOURCES:
    if time.monotonic()-START>TIME_LIMIT_HOURS*3600:
        print('Runtime budget reached before starting',source);break
    statuses[source]=train_one(source)
    if statuses[source]!='COMPLETED':break
print('R04 run statuses:',statuses)

R04 rotation phase=head epoch=1/8 trainable=16,392
rotation head 01/8 | train CE 0.4331 | val BACC 0.2289 | macro-F1 0.1727 | val NLL 1.3976 | 161s
rotation head 02/8 | train CE 0.4186 | val BACC 0.2367 | macro-F1 0.1856 | val NLL 1.4839 | 29s
rotation head 03/8 | train CE 0.4123 | val BACC 0.2479 | macro-F1 0.2135 | val NLL 1.3119 | 30s
rotation head 04/8 | train CE 0.4111 | val BACC 0.2438 | macro-F1 0.2285 | val NLL 1.2375 | 29s
rotation head 05/8 | train CE 0.4118 | val BACC 0.2164 | macro-F1 0.2077 | val NLL 1.1861 | 29s
rotation head 06/8 | train CE 0.4082 | val BACC 0.2068 | macro-F1 0.1943 | val NLL 1.2535 | 29s
rotation head 07/8 | train CE 0.4071 | val BACC 0.2242 | macro-F1 0.2036 | val NLL 1.2849 | 29s
rotation head 08/8 | train CE 0.4085 | val BACC 0.2601 | macro-F1 0.2415 | val NLL 1.2692 | 29s
R04 rotation phase=finetune epoch=1/35 trainable=23,471,304
rotation finetune 01/35 | train CE 0.3190 | val BACC 0.4102 | macro-F1 0.3852 | val NLL 0.9937 | 73s *
rotation finetune

## 4. Consolidate artifacts and show next-stage requirements

In [5]:
result=[]
for source in SOURCES:
    path=OUT/'reports'/source/'final_metrics.json'
    if path.exists():
        obj=json.loads(path.read_text())
        result.append({'source':source,'seed':SEED,'status':'COMPLETED',
            'best_epoch':obj['best_finetune_epoch'],
            'validation_balanced_accuracy':obj['validation_metrics']['balanced_accuracy'],
            'validation_macro_f1':obj['validation_metrics']['macro_f1'],
            'validation_accuracy':obj['validation_metrics']['accuracy'],
            'validation_nll':obj['validation_metrics']['nll']})
    else:result.append({'source':source,'seed':SEED,'status':statuses.get(source,'NOT_STARTED')})
result_df=pd.DataFrame(result)
result_df.to_csv(OUT/'tables/summary.csv',index=False)
print(result_df.to_string(index=False))
status='COMPLETED' if all(row['status']=='COMPLETED' for row in result) else 'PAUSED'
save_json(OUT/'reports/run_summary.json',{'stage':'R04','seed':SEED,'status':status,
          'model_status':{s:next(row['status'] for row in result if row['source']==s) for s in SOURCES},
          'n01_manifest_sha256':protocol['n01_manifest_sha256'],
          'elapsed_seconds':time.monotonic()-START,
          'required_next_stage':'R05 feature bank: models/rotation/encoder_best.pt and models/simclr/encoder_best.pt',
          'r07_predictions':['predictions/rotation/validation.npz','predictions/rotation/meta_train.npz',
                              'predictions/simclr/validation.npz','predictions/simclr/meta_train.npz']})
files=[{'path':p.relative_to(OUT).as_posix(),'size_bytes':p.stat().st_size} for p in OUT.rglob('*') if p.is_file() and p.name!='artifact_manifest.json']
save_json(OUT/'reports/artifact_manifest.json',{'stage':'R04','seed':SEED,'status':status,
          'files':sorted(files,key=lambda x:x['path'])})
print('R04:',status,'Output:',OUT)
if status!='COMPLETED':print('Attach this R04 Kaggle output to the next run of the same seed to resume safely.')
else:print('Keep this completed Kaggle notebook output attached for R05 and R07; test data was not used.')

  source  seed    status  best_epoch  validation_balanced_accuracy  validation_macro_f1  validation_accuracy  validation_nll
rotation    42 COMPLETED          10                      0.577174             0.527038             0.705855        0.850625
  simclr    42 COMPLETED          14                      0.575618             0.534331             0.719579        1.101292
R04: COMPLETED Output: /kaggle/working/lesion-r04-ssl-classification-s42
Keep this completed Kaggle notebook output attached for R05 and R07; test data was not used.
